# 04 · ML Model Training

**Round 1 walkthrough:** train and compare five distinct algorithms. A whole-trip holdout is used so observations from a trip do not appear in both fitting and validation. The student-test partition is kept for the permitted development comparison.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
DATA=ROOT/"data"/"Student folder"
print("Project:",ROOT,"| data folder available:",DATA.exists())

Project: /Users/yashas/Documents/Codex/2026-11-18/referenced-chatgpt-conversation-this-is-an-2/ML_hackathon | data folder available: True


The target is binary: a candidate is positive when it matches the closest arc in that trip’s supplied reference route. Arc IDs are trip-local, so the model learns general geometry and movement patterns rather than one global road-ID vocabulary. Candidate-miss observations are counted and omitted from ranker fitting because the correct arc is not available to choose.

In [2]:
from pathsnap.matcher import train_compare
rows,selected_model,fitted_model=train_compare(DATA,ROOT/"outputs",k=12,seed=42,max_candidates=200_000)
print("Validation-selected model:",selected_model)
print("Saved final fit:",ROOT/"outputs"/"phase1_model.joblib")

Validation-selected model: Hist Gradient Boosting
Saved final fit: /Users/yashas/Documents/Codex/2026-11-18/referenced-chatgpt-conversation-this-is-an-2/ML_hackathon/outputs/phase1_model.joblib


In [3]:
import pandas as pd
comparison=pd.read_csv(ROOT/"outputs"/"model_comparison.csv")
columns=["model","training_seconds","validation_top1_accuracy","validation_candidate_macro_f1","student_test_top1_accuracy","student_test_candidate_macro_f1","student_test_candidate_log_loss","student_test_inference_seconds"]
display(comparison[columns].style.format({c:"{:.4f}" for c in columns if c!="model"}))

,model,training_seconds,validation_top1_accuracy,validation_candidate_macro_f1,student_test_top1_accuracy,student_test_candidate_macro_f1,student_test_candidate_log_loss,student_test_inference_seconds
0,Nearest road segment,nan,0.5537,nan,0.5348,nan,nan,nan
1,Logistic Regression,0.0477,0.5522,0.6301,0.5650,0.6022,0.4722,0.0047
2,KNN,0.0014,0.6079,0.6965,0.6569,0.7251,0.5413,0.4709
3,Decision Tree,0.5822,0.6887,0.7989,0.7294,0.8024,0.2230,0.0076
4,Random Forest,2.3749,0.7109,0.8323,0.7514,0.8414,0.1385,0.1834
5,Hist Gradient Boosting,0.5146,0.7145,0.8478,0.7545,0.8609,0.1065,0.1146


## Configurations shown in this run

- Logistic Regression: standardized features, max 250 iterations, balanced classes.
- KNN: standardized features, 9 distance-weighted neighbors, 5,000 balanced candidate rows for practical inference.
- Decision Tree: depth 14, minimum 4 rows per leaf, balanced classes.
- Random Forest: 80 trees, depth 18, minimum 2 rows per leaf, balanced subsampling.
- Hist Gradient Boosting: 100 iterations, learning rate 0.08, 15 maximum leaf nodes, L2 regularization 1.0.